In [60]:
import os
import pandas as pd
data = os.path.join("Data", "SAIL2025_LVMA_data_3min_20August-25August2025_flow.csv") 
sensor_loc = os.path.join("Data", "sensor-location.csv") 
df_data = pd.read_csv(data)
df_sensors = pd.read_csv(sensor_loc, sep =  ";")

base_names = set()
for col in df_data.columns:
    if "_" in col:
        base = "_".join(col.split("_")[:-1])
        base_names.add(base)

time_cols = ["timestamp", "hour", "minute", "day", "month", "weekday", "is_weekend"]
time_cols = [c for c in time_cols if c in df_data.columns]  # only keep existing ones
df_combined = df_data[time_cols].copy()

#here we combine the data
for base in base_names:
    related_cols = [c for c in df_data.columns if c.startswith(base + "_")]
    if len(related_cols) >1: 
        df_combined[base] = df_data[related_cols].sum(axis=1)   
print(df_sensors)


     Objectummer                        Locatienaam             Lat/Long  \
0   CMSA-GAKH-01              Kalverstraat t.h.v. 1  52.372634, 4.892071   
1   CMSA-GAWW-11                       Korte Niezel  52.374616, 4.899830   
2   CMSA-GAWW-12                    Oudekennissteeg  52.373860, 4.898690   
3   CMSA-GAWW-13                         Stoofsteeg  52.372439, 4.897689   
4   CMSA-GAWW-14    Oudezijds Voorburgwal t.h.v. 91  52.373538, 4.898166   
5   CMSA-GAWW-15  Oudezijds Achterburgwal t.h.v. 86  52.372916, 4.898207   
6   CMSA-GAWW-16  Oudezijds Achterburgwal t.h.v. 91  52.372628, 4.898233   
7   CMSA-GAWW-17   Oudezijds Voorburgwal t.h.v. 206  52.372782, 4.896649   
8   CMSA-GAWW-19                         Molensteeg  52.373587, 4.899815   
9   CMSA-GAWW-20                      Oudebrugsteeg  52.375350, 4.897480   
10  CMSA-GAWW-21                          Damstraat  52.371930, 4.895600   
11  CMSA-GAWW-23                        Bloedstraat  52.372764, 4.899829   
12       GAC

In [56]:
#effectieve breedte bruikbaar maken om mee te rekenen
df_sensors["Effectieve breedte"] = (
    df_sensors["Effectieve breedte"]
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)

#directional flow
directional_cols = [c for c in df_data.columns if "_" in c]
id_cols = ["timestamp", "hour", "minute", "day", "month", "weekday", "is_weekend"]

df_long = df_data.melt(
    id_vars=id_cols,
    value_vars=directional_cols,
    var_name="sensor_direction",
    value_name="count"
)

df_long["Objectummer"] = df_long["sensor_direction"].str.rsplit("_", n=1).str[0]


df_long = df_long.merge(
    df_sensors[["Objectummer", "Effectieve breedte"]],
    on="Objectummer",
    how="left"
)

df_long["flow"] = df_long["count"] / (3 * df_long["Effectieve breedte"]) # flow = count /( 3 (min)  * effective witdh field)

df_long_sorted = df_long.sort_values(by="flow", ascending=False)


#combined flow
df_combined_long = (
    df_long
    .groupby(["timestamp", "Objectummer"], as_index=False)
    .agg({"flow": "sum"})
)

df_combined_long_sorted = df_combined_long.sort_values(by="flow", ascending=False)


In [62]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import plotly.express as px


# Ensure timestamp is datetime
df_long_sorted["timestamp"] = pd.to_datetime(df_long_sorted["timestamp"])
sensors = df_sensors["Objectummer"]
for sensor in sensors:
    df_plot = df_long_sorted[df_long_sorted["Objectummer"] == sensor]

    plt.figure(figsize=(12,5))
    sns.lineplot(
        data=df_plot,
        x="timestamp",
        y="flow",
        hue="sensor_direction",   # each direction gets its own line
        marker="o"
    )
    plt.title(f"Directional flow over time for {sensor}")
    plt.xlabel("Time")
    plt.ylabel("Flow (person/min/meter)")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    # plt.show()
    # os.makedirs("plots", exist_ok=True)
    plt.savefig(f"plots/{sensor}.png")
    plt.close() 


In [74]:
import base64
import folium
from folium import IFrame


# Center map roughly at sensor locations
df_sensors[['latitude', 'longitude']] = df_sensors['Lat/Long'].str.split(',', expand=True)
df_sensors['latitude'] = pd.to_numeric(df_sensors['latitude'])
df_sensors['longitude'] = pd.to_numeric(df_sensors['longitude'])


map_center = [df_sensors['latitude'].mean(), df_sensors['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=15)

# Add zones as colored polygons
# print(gdf_zones)
gdf_zones = gdf_zones.to_crs(epsg=4326)  # ensure lat/lon
# print(gdf_zones)
folium.GeoJson(
    gdf_zones,
    style_function=lambda feature: {
        'fillColor': feature['properties']['color'],
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.5
    }
).add_to(m)

# Add sensor markers with popups showing the plots
for idx, row in df_sensors.iterrows():
    sensor = row['Objectummer']
    plot_path = f"plots/{sensor}.png"
    
    if os.path.exists(plot_path):
        encoded = base64.b64encode(open(plot_path, 'rb').read()).decode()
        html = f'<img src="data:image/png;base64,{encoded}" width="800" height="350">'
        iframe = IFrame(html, width=800, height=330)
        popup = folium.Popup(iframe, max_width=1000)
    else:
        popup = None
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7,
        color='red',
        fill=True,
        fill_color='red',
        popup=popup
    ).add_to(m)

# Save map to HTML
# m
m.save("interactive_sensor_map.html")